In [ ]:
import sys
from pathlib import Path
import numpy as np

src = str(Path("E:/Agent/koopman_sciml/src"))
if src not in sys.path:
    sys.path.insert(0, src)

from oat15a_pod import PodConfig, resolve_data_dir

data_dir = resolve_data_dir(
    PodConfig(data_root="F:/MY JOURNEY/data-driven_model")
)
P_full = np.loadtxt(data_dir / "Snapshotp.dat")
P = P_full[:, ::2]
P_mean = P.mean(axis=1, keepdims=True)
X = P - P_mean

print("原始压力：", P_full.shape)
print("抽样压力：", P.shape)
print("时间均值：", P_mean.shape)
print("压力脉动：", X.shape)

In [ ]:
U, s, Vh = np.linalg.svd(X, full_matrices=False)
total_energy = np.sum(s**2)
assert total_energy > 0

print("阶数 | 累计能量 | 实际重构误差 | 理论误差 | 两者差值")

for r in [1, 3, 5, 8, 10]:
    energy = np.sum(s[:r]**2) / total_energy
    X_r = (U[:, :r] * s[:r]) @ Vh[:r, :]

    actual = np.linalg.norm(X - X_r, "fro") / np.linalg.norm(X, "fro")
    predicted = np.sqrt(max(0.0, 1.0 - energy))

    print(
        f"{r:2d} | {energy:.6%} | {actual:.6%} | "
        f"{predicted:.6%} | {abs(actual - predicted):.2e}"
    )
    assert np.isclose(actual, predicted, rtol=1e-8, atol=1e-10)